In [ ]:
import polars as pl
import pandas as pd

import sys
sys.path.append('../../04_utils')

from utils import name_finder, low_context_name_finder

In [ ]:
# Define pathings
data_path = '../../01_data/'
out_path = '../../03_output/01_enriched_results/'

In [3]:
#Load data
verbs_df = pl.read_csv(data_path + 'DS1 modal verbs.csv', separator= ',' )

verbs_df.head()

Left,KWIC,Right
str,str,str
"""<s> NARRATOR Yes, indeed. </s>…","""shall""","""be chosen to leave the Undead …"
"""Artorias art none but a fabric…","""Wilt""","""thou not join us? </s><s> Oh y…"
"""the dark? </s><s> ''Tis but a …","""would""","""suit thee well. </s><s> Hmm, I…"
"""suit thee well. </s><s> Hmm, I…","""will""","""summon thee. </s><s> Fend them…"
""". </s><s> Then there is nothin…","""shall""","""summon others, who will by the…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
# Open raw text to look for missing verbs
with open(data_path + 'Ds corpus input.txt', 'r', encoding='utf-8') as file:
        raw_text = file.read()  # Read the entire content into a single string
        # print(raw_text)

In [6]:
# Clean the raw text to facilitate matching with low_context_name_finder
punctuation = [',','.',';',':','!','?']

raw_text_clean = raw_text.replace('\n',' ').replace('…','...').replace('‘',"'").replace("' ","'").strip()
for punct in punctuation:
        raw_text_clean = raw_text_clean.replace(punct, punct + ' ').replace(punct,'')

raw_text_clean = raw_text_clean.split(' ')

raw_text_clean = [word for word in raw_text_clean if word != '']

raw_text_clean = ' '.join(raw_text_clean)

# raw_text_clean

In [ ]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values.
verbs_df = verbs_df.with_columns(pl.col('Left').map_elements(lambda x: low_context_name_finder(x, raw_text_clean, 100), 
                                                             return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\

verbs_df

Left,KWIC,Right,Character
str,str,str,str
""" NARRATOR Yes, indeed. The D…","""shall""","""be chosen to leave the Undead …","""NARRATOR"""
"""Artorias art none but a fabric…","""wilt""","""thou not join us? Oh yes, I …","""ALVINA OF THE DARKROOT WOOD"""
"""the dark? ''Tis but a fairy …","""would""","""suit thee well. Hmm, I see. …","""ALVINA OF THE DARKROOT WOOD"""
"""suit thee well. Hmm, I see. …","""will""","""summon thee. Fend them off s…","""ALVINA OF THE DARKROOT WOOD"""
""". Then there is nothing more…","""shall""","""summon others, who will by the…","""ALVINA OF THE DARKROOT WOOD"""
…,…,…,…
"""THOROLUND Hm? What have we h…","""ca""","""n''t very well abandon them no…","""VINCE OF THOROLUND"""
"""! Oh, you again. What busi…","""can""","""do. I am Vince of Thorolund.…","""VINCE OF THOROLUND"""
""". Oh, you yet again. You''…","""can""","""one do? I do hope we meet ag…","""VINCE OF THOROLUND"""


In [8]:
verbs_df.filter(pl.col('Character').is_null())#['Left'].to_list()

Left,KWIC,Right,Character
str,str,str,str


In [10]:
#Add character class and age information from master table
verbs_df = verbs_df.join(char_master_df, on='Character', how = 'left')

verbs_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
""" NARRATOR Yes, indeed. The D…","""shall""","""be chosen to leave the Undead …","""NARRATOR""",null,null
"""Artorias art none but a fabric…","""wilt""","""thou not join us? Oh yes, I …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""the dark? ''Tis but a fairy …","""would""","""suit thee well. Hmm, I see. …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""suit thee well. Hmm, I see. …","""will""","""summon thee. Fend them off s…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
""". Then there is nothing more…","""shall""","""summon others, who will by the…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
…,…,…,…,…,…
"""THOROLUND Hm? What have we h…","""ca""","""n''t very well abandon them no…","""VINCE OF THOROLUND""","""Low""","""Young"""
"""! Oh, you again. What busi…","""can""","""do. I am Vince of Thorolund.…","""VINCE OF THOROLUND""","""Low""","""Young"""
""". Oh, you yet again. You''…","""can""","""one do? I do hope we meet ag…","""VINCE OF THOROLUND""","""Low""","""Young"""


In [11]:
verbs_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str


In [12]:
# Sanity check for extracted characters
verbs_df.to_pandas()['Character'].unique()

array(['NARRATOR', 'ALVINA OF THE DARKROOT WOOD', 'ANASTACIA OF ASTORA',
       'ANDRE OF ASTORA', 'BIG HAТ LOGAN', 'BLACKSMITH VAMOS',
       'CRESTFALLEN MERCHANT', 'CRESTFALLEN WARRIOR',
       'CROSSBREED PRISCILLA', 'DARK SUN GWYNDOLIN', 'DARKMOON KNIGHTESS',
       'DARKSTALKER KAATHЕ', 'DOMHNALL OF ZENA', 'DUSK OF OOLACILE',
       'EINGYI OF THE GREAT SWAMP', 'ELIZABETH KEEPER OF THE SANCTUARY',
       'GIANT BLACKSMITH', 'GRIGGS OF VINHEIM',
       'GWYNEVERE PRINCESS OF SUNLIGHT', 'HAWKEYE GOUGН',
       'INGWARD KEEPER OF THE SEAL', 'KINGSEEKER FRAMPТ',
       'LAURENTIUS OF THЕ GREAT SWAMP', 'LAUTREC OF CARIM',
       "LORD''S BLADE CIARAN", "LORD'S BLADE CIARAN", 'MARVELOUS CHESTER',
       'OSCAR OF ASTORA', 'OSWALD OF CARIM', '-I', 'PETRUS OF THOROLUND',
       'QUELANA OF IZALITH', 'RHEA OF THOROLUND', 'RICKERT OF VINHEIM',
       'SHIVA OF THE EAST', 'SIEGLINDE OF CATARINA',
       'SIEGMEYER OF CATARINA', 'SOLAIRE OF ASTORA', 'THE FAIR LADY',
       'TRUSTY PATCHES', 

In [14]:
# Save enriched dataframe into a csv file.
verbs_df.write_csv(out_path + 'modal_verbs_analysis.csv', separator= ';')